# Pilot-starved OFDM: problem and signal

One reference symbol must support eight subsequent data symbols. Fewer pilots leave more room for payload, but noisy synchronization estimates have to remain useful across a longer data segment.

**Frame:** `| pilot | data 1 | data 2 | … | data 8 |`  
Each symbol has 64 FFT samples + 16 CP samples: pilot indices 0–79, data indices 80–719.

## Signal chain

1024 information bits → 5G LDPC → 2048 coded bits → 512 16-QAM symbols → eight OFDM data symbols. A deterministic QPSK pilot OFDM symbol precedes the data.

The synthetic channel adds a frame-start phase offset, constant CFO, and AWGN. There is no multipath channel in this experiment.

## Phase offset and carrier frequency offset

Phase offset rotates the signal; CFO makes that rotation accumulate with time:

$$y[t]=x[t]e^{j(\phi_0+2\pi f t)}+w[t].$$

Here $f$ is cycles/sample and $\phi_0$ is radians at full-frame sample zero. Even a small CFO estimation error leaves increasing residual phase error on later data samples. The classical receiver estimates both parameters using only the initial pilot.

## A representative historical frame

![Historical time-domain OFDM frame](../results/figures/signal_12db.png)

Historical I-channel example at 12 dB, CFO fixed at +2e−4 cycles/sample, seed 46. Orange is transmitted I; blue is received I. The red line marks the pilot's end, and the other boundaries separate OFDM symbols. This is an illustration of impairment, not a receiver result.

Source: `0K_Dataset_Examples.ipynb`, cell 2, eighth PNG output, commit `a5a14588054df1e644b010cbc929c19f1f81d392`. The image was reused without regenerating data. The constellation image was omitted because it pools QPSK pilot and 16-QAM data points.

## Physics-informed reception

The estimator uses both pilot and data I/Q. Known physics supplies inverse rotation at the correct full-frame time indices; CP removal and FFT then expose frequency-domain features. Residual Conv1D layers and FiLM-conditioned LinearAttention refine those features into bit logits. LDPC supplies the final information bits.

The auxiliary I/Q and phase/CFO regression losses guide training. Physical units and numerical CFO scaling are distinct. The reusable correction is `compensate_data` in [models.py](../src/lightweight_receiver/models.py); simulation lives in [simulation.py](../src/lightweight_receiver/simulation.py).

Continue to [Results and comparison](02_results_and_comparison.ipynb). No model training or signal generation is needed to read these notebooks.